# Moltiplicare due tabelle: righe per colonne

*Anatomia di un LLM* · capitolo 3, I mattoni · script `01-09`, `scripts/01_09_moltiplicare_matrici.py`

La moltiplicazione fra matrici e' un prodotto scalare ripetuto: ogni casella
del risultato e' una riga della prima tabella per una colonna della seconda.
E' l'operazione che il modello fa piu' di ogni altra, ed e' quella che
l'attenzione usa per confrontare tutte le parole con tutte le altre in un
colpo solo.

Si esegue con `python3 scripts/01_09_moltiplicare_matrici.py`
e dura circa 5 secondi.

> **Per riottenere i numeri del libro** esegui le celle in ordine, dalla prima. Il seme si fissa una volta sola, in `prepara()`: rieseguire una cella a metà cambia i numeri casuali di tutte quelle dopo.

[Apri in Colab](https://colab.research.google.com/github/fcollova/llm-anatomy/blob/main/notebooks/01_09_moltiplicare_matrici.ipynb) · la prima cella scarica il codice, se non lo trova accanto.

In [1]:
# Porta il notebook nella cartella che contiene llm_anatomy/, da dovunque
# sia stato aperto: gli script si eseguono da li', e cosi' anche le celle.
import os
import subprocess
import sys
from pathlib import Path

radice = next((p for p in [Path.cwd(), *Path.cwd().parents]
               if (p / "llm_anatomy").is_dir()), None)

if radice is None:
    # Il notebook e' stato aperto da solo, per esempio su Colab: il codice, il
    # testo e il modello addestrato si scaricano una volta, qui accanto.
    REPO, RAMO, CARTELLA = "fcollova/llm-anatomy", "main", ""
    copia = Path("anatomia-di-un-llm")
    if not copia.exists():
        esito = subprocess.run(
            ["git", "clone", "-q", "--depth", "1", "--branch", RAMO,
             f"https://github.com/{REPO}.git", str(copia)], capture_output=True)
        if esito.returncode != 0:
            raise SystemExit("Non riesco a scaricare il codice da github.com/"
                             + REPO + ": " + esito.stderr.decode()[-300:])
    radice = copia / CARTELLA

os.chdir(radice)
sys.path.insert(0, str(Path.cwd()))

In [2]:
import torch

from llm_anatomy.utils import prepara, stampa_shape, titolo, verifica

prepara()

[ambiente] torch 2.13.0+cpu  device=cpu  threads=8  seed=1337


device(type='cpu')

In [3]:
titolo("Righe per colonne, a mano")
A = torch.tensor([[1.0, 2.0, 3.0],
                  [4.0, 5.0, 6.0]])
M = torch.tensor([[1.0, 0.0],
                  [0.0, 1.0],
                  [1.0, 1.0]])
P = A @ M
stampa_shape("A", A)
stampa_shape("M", M)
stampa_shape("A @ M", P)
print()
print(P)
riga, colonna = A[0], M[:, 1]
print(f"\n  casella (0, 1): riga 0 di A {riga.tolist()}, colonna 1 di M "
      f"{colonna.tolist()}")
print(f"  1*0 + 2*1 + 3*1 = {torch.dot(riga, colonna).item():.0f}")
verifica(all(P[i, j] == torch.dot(A[i], M[:, j])
             for i in range(2) for j in range(2)),
         "ogni casella e' il prodotto scalare di una riga per una colonna")
print("  le lunghezze interne devono coincidere: (2, 3) @ (3, 2) -> (2, 2)")


Righe per colonne, a mano
-------------------------
  A                            (2, 3)                 float32
  M                            (3, 2)                 float32
  A @ M                        (2, 2)                 float32

tensor([[ 4.,  5.],
        [10., 11.]])

  casella (0, 1): riga 0 di A [1.0, 2.0, 3.0], colonna 1 di M [0.0, 1.0, 1.0]
  1*0 + 2*1 + 3*1 = 5
  [verifica superata] ogni casella e' il prodotto scalare di una riga per una
                      colonna
  le lunghezze interne devono coincidere: (2, 3) @ (3, 2) -> (2, 2)


In [4]:
titolo("Tutte le coppie in un colpo solo")
T, d = 4, 8
q = torch.randn(T, d)
k = torch.randn(T, d)
punteggi = q @ k.T
print(f"  {T} domande e {T} risposte, ognuna lunga {d} numeri")
print(f"  q @ k.T da' una tabella {tuple(punteggi.shape)}: "
      "una casella per coppia\n")
for i in range(T):
    print("   " + " ".join(f"{v:7.3f}" for v in punteggi[i].tolist()))
a_mano = torch.zeros(T, T)
for i in range(T):
    for j in range(T):
        a_mano[i, j] = torch.dot(q[i], k[j])
verifica(torch.allclose(a_mano, punteggi, atol=1e-6),
         f"le {T * T} caselle coincidono con {T * T} prodotti scalari "
         "fatti a mano")
print("  k.T e' k con righe e colonne scambiate: serve perche' le lunghezze")
print("  interne coincidano, (4, 8) @ (8, 4)")


Tutte le coppie in un colpo solo
--------------------------------
  4 domande e 4 risposte, ognuna lunga 8 numeri
  q @ k.T da' una tabella (4, 4): una casella per coppia

    -1.569  -0.210   1.753   2.303
    -1.591  -2.180  -8.129  -4.529
    -2.109  -0.402   9.644   3.524
    -4.248  -2.402  -2.549   2.222
  [verifica superata] le 16 caselle coincidono con 16 prodotti scalari fatti a
                      mano
  k.T e' k con righe e colonne scambiate: serve perche' le lunghezze
  interne coincidano, (4, 8) @ (8, 4)


## Da provare

scambia l'ordine, M @ A, e leggi l'errore: dice quali lunghezze non tornano.